# RAG - Multi-Step Reasoning

## dependencies

In [ ]:
from typing import List, Dict, Any, Optional, TypedDict
from dataclasses import dataclass, field
from langchain.schema import Document
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

import os
os.makedirs("logs", exist_ok=True)
import sys
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/rag.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()

## constants and type definitions

In [ ]:
CHAT_MODEL = 'gemini-2.5-flash'
REWRITE_MODEL = 'gemini-2.5-flash'
EMBEDDING_MODEL = 'models/embedding-001'

MAX_REPHRASE_ATTEMPTS = 2
RETRIEVAL_K = 3

TOPIC_LIST = [
    "Gym History & Founder",
    "Operating Hours",
    "Membership Plans",
    "Fitness Classes",
    "Personal Trainers",
    "Facilities & Equipment",
    "Anything else about Jiten's Gym",
]

CLASSIFICATION_SYSTEM_PROMPT = (
    "You are a domain classifier for a gym knowledge base. "
    "Answer ONLY 'Yes' or 'No'. A question is ON-TOPIC iff it relates to: "
    + ", ".join(TOPIC_LIST)
)

REWRITE_SYSTEM_PROMPT = (
    "Rewrite the user question for optimal semantic vector retrieval in a gym knowledge base. "
    "Return ONLY the rewritten question. Do NOT answer it."
)

DOC_GRADE_SYSTEM_PROMPT = (
    "You are a strict document relevance grader. Respond ONLY 'Yes' or 'No'. "
    "Answer 'Yes' only if the document directly helps answer the question."
)

ANSWER_SYSTEM_PROMPT = (
    "You are an assistant for Jiten's Gym. Use ONLY the provided documents. "
    "If the documents lack sufficient info, state that briefly."
)

## schema / state definitions

In [ ]:
class AgentState(TypedDict):
    messages: List[BaseMessage]
    documents: List[Document]
    on_topic: str
    rephrased_question: str
    proceed_to_generate: bool
    rephrase_count: int
    question: HumanMessage

@dataclass
class RAGConfig:
    chat_model: str = CHAT_MODEL
    rewrite_model: str = REWRITE_MODEL
    embed_model: str = EMBEDDING_MODEL
    retrieval_k: int = RETRIEVAL_K
    max_rephrase_attempts: int = MAX_REPHRASE_ATTEMPTS
    # enable_debug: bool = os.getenv("DEBUG_MODE", "false").lower() == "true"
    documents: List[Document] = field(default_factory=list)

## tool definitions (helpers and factories)

In [ ]:
def build_embeddings(model_name: str) -> GoogleGenerativeAIEmbeddings:
    try:
        logger.debug("🧠 Init embeddings model=%s", model_name)
        return GoogleGenerativeAIEmbeddings(model=model_name)
    except Exception as exc:
        logger.exception("❌ Embeddings init failed: %s", exc)
        raise

def build_chat_model(model_name: str, temperature: float = 0.1) -> ChatGoogleGenerativeAI:
    try:
        logger.debug("🧠 Init chat model=%s", model_name)
        return ChatGoogleGenerativeAI(model=model_name, temperature=temperature)
    except Exception as exc:
        logger.exception("❌ Chat init failed: %s", exc)
        raise

def parse_yes_no(text: str) -> str:
    return text.strip().lower().startswith("y")

## main class

In [ ]:
class RAGPipeline:
    """End-to-end RAG pipeline with classification, rewrite, retrieval, grading, and answer generation."""

    def __init__(
            self,
            config: RAGConfig,
            persist_dir: Optional[str] = None,
            enable_vector_persist: bool = True,
        ):
        self.config = config
        self.persist_dir = persist_dir
        self.enable_vector_persist = enable_vector_persist

        logger.info(f"🚀 Init GymRAGPipeline chat={config.chat_model} embed={config.embed_model} k={config.retrieval_k}")

        # llm instances
        self.llm_chat = build_chat_model(config.chat_model, temperature=0.1)
        self.llm_rewrite = build_chat_model(config.rewrite_model, temperature=0.1)

        # embeddings and vector store
        self.embeddings = build_embeddings(config.embed_model)
        self.vectorstore: Optional[Chroma] = None
        self.retriever = None

        # langgraph
        self._graph = None
        self._app = None
        self.memory = MemorySaver()
        self._build_graph()

    # core methods
    def add_documents(self, docs: List[Document]) -> None:
        if not docs:
            logger.warning("⚠️ No documents provided to add.")
            return
        logger.info(f"🧩 Adding {len(docs)} documents.")
        self.config.documents = docs
        self._build_vectorstore()

    def ask(self, question: str, thread_id: str = '1') -> str:
        if not question.strip():
            logger.warning("⚠️ Empty question provided.")
            return "Please provide a valid question."
        if self.retriever is None:
            if self.config.documents:
                self._build_vectorstore()
            else:
                logger.error("❌ No documents in vectorstore. Cannot proceed.")
                return "No documents available to answer the question."
        
        state: AgentState = {
            "messages": [],
            "documents": [],
            "on_topic": "",
            "rephrased_question": "",
            "proceed_to_generate": False,
            "rephrase_count": 0,
            "question": HumanMessage(content=question)
        }
        try:
            config = {'configurable': {'thread_id': thread_id}}

            logger.info(f"💬 Processing question: {question}")
            result = self._app.invoke(state, config=config)
            msgs: List[BaseMessage] = result.get("messages", [])
            answer = self._extract_last_ai_message(msgs)
            logger.info(f"✅ Answer generated: {answer}")
            return answer
        except Exception as e:
            logger.exception(f"❌ Error during ask: {e}")
            return "An error occurred while processing your question."
    
    def visualize(self, mode: str = 'auto'):
        try:
            if self._app is None and self._graph is None:
                logger.warning("⚠️ Graph not built yet.")
                return

            graph_obj_getter = getattr(self._app, 'get_graph', None)
            graph_obj = graph_obj_getter() if callable(graph_obj_getter) else self._graph
            
            # 2. Mermaid source ---------------------------------------------------
            if mode in ('auto', 'mermaid') and hasattr(graph_obj, 'draw_mermaid'):
                try:
                    mermaid_src = graph_obj.draw_mermaid()
                    print(mermaid_src)
                    logger.info("🧪 Mermaid diagram (text) printed.")
                    if mode != 'auto':
                        return
                except Exception as e:
                    logger.debug("⚠️ Mermaid render not available: %s", e)

            # 3. ASCII fallback ---------------------------------------------------
            if mode in ('auto', 'ascii') and hasattr(graph_obj, 'draw_ascii'):
                try:
                    ascii_map = graph_obj.draw_ascii()
                    print(ascii_map)
                    logger.info("📄 ASCII graph printed.")
                    return
                except Exception as e:
                    logger.debug("⚠️ ASCII render not available: %s", e)

            logger.warning("⚠️ No supported visualization method found (mode=%s).", mode)
        except Exception as e:
            logger.exception(f"❌ Visualization error: {e}")
    
    # internal build methods
    def _build_vectorstore(self) -> None:
        try:
            persist_args: Dict[str, Any] = {}
            if self.enable_vector_persist and self.persist_dir:
                os.makedirs(self.persist_dir, exist_ok=True)
                persist_args['persist_directory'] = self.persist_dir
                logger.info(f"💾 Vectorstore will persist to {self.persist_dir}")
            
            # Check if vector store already exists
            if self.enable_vector_persist and self.persist_dir and os.path.exists(self.persist_dir):
                try:
                    # Try loading existing vector store
                    self.vectorstore = Chroma(
                        embedding_function=self.embeddings,
                        persist_directory=self.persist_dir
                    )
                    # Verify it has documents
                    if self.vectorstore._collection.count() > 0:
                        logger.info(f"📦 Loaded existing vectorstore with {self.vectorstore._collection.count()} documents.")
                        self.retriever = self.vectorstore.as_retriever(search_kwargs={'k': self.config.retrieval_k})
                        return
                    else:
                        logger.info("📦 Existing vectorstore is empty, rebuilding...")
                except Exception as e:
                    logger.warning(f"⚠️ Could not load existing vectorstore: {e}. Creating new one...")
            
            # Create new vector store from documents
            self.vectorstore = Chroma.from_documents(
                documents=self.config.documents,
                embedding=self.embeddings,
                **persist_args
            )
            self.retriever = self.vectorstore.as_retriever(search_kwargs={'k': self.config.retrieval_k})
            logger.info(f"📦 Vectorstore built successfully with {len(self.config.documents)} docs.")
        except Exception as e:
            logger.exception(f"❌ Vectorstore build failed: {e}")
            raise

    def _build_graph(self) -> None:
        logger.debug("🔧 Building LangGraph pipeline.")
        try:
            graph = StateGraph(AgentState)

            # nodes
            graph.add_node('rewrite_question', self._node_rewrite_question)
            graph.add_node('classify_question', self._node_classify_question)
            graph.add_node('retrieve_documents', self._node_retrieve_documents)
            graph.add_node('grade_documents', self._node_grade_documents)
            graph.add_node('generate_answer', self._node_generate_answer)
            graph.add_node('generate_fallback', self._node_generate_fallback)
            graph.add_node('off_topic', self._node_off_topic)

            # entry point
            graph.set_entry_point('classify_question')

            # edges
            graph.add_conditional_edges(
                'classify_question', 
                self._route_after_classification, 
                {'on_topic': 'retrieve_documents', 'off_topic': 'off_topic'}
            )

            graph.add_edge('retrieve_documents', 'grade_documents')

            graph.add_conditional_edges(
                'grade_documents',
                self._route_after_grading,
                {'generate': 'generate_answer', 'fallback': 'generate_fallback', 'refine': 'rewrite_question'}
            )

            graph.add_edge('generate_answer', END)
            graph.add_edge('generate_fallback', END)
            graph.add_edge('off_topic', END)
            graph.add_edge('rewrite_question', 'retrieve_documents')

            self._graph = graph
            self._app = graph.compile(checkpointer=self.memory)
            logger.info("🛠️ LangGraph pipeline built successfully.")
        except Exception as e:
            logger.exception(f"❌ Graph build failed: {e}")
            raise
    # graph routing methods 
    def _route_after_classification(self, state: AgentState) -> str:
        return 'on_topic' if state.get('on_topic') == 'Yes' else 'off_topic'

    def _route_after_grading(self, state: AgentState) -> str:
        if state.get('proceed_to_generate'):
            logger.debug("➡️ Proceeding to generate answer.")
            return 'generate'
        elif state.get('rephrase_count', 0) < self.config.max_rephrase_attempts:
            logger.debug(f"🔄 Rephrasing question and retrying retrieval (attempt: {state.get('rephrase_count', 0)}).")
            return 'rewrite_question'
        else:
            logger.debug(f"❗ Max rephrase attempts reached. Falling back (attempt: {state.get('rephrase_count', 0)}).")
            return 'fallback'

    # node implementations
    def _node_rewrite_question(self, state: AgentState) -> AgentState:
        base = state.get('rephrased_question') or state['question'].content 
        attempt = state.get('rephrase_count', 0) + 1
        logger.debug(f"✍️ Rewriting question, attempt {attempt}: {base}")
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", REWRITE_SYSTEM_PROMPT),
                ("user", "Rewrite this question for vector retrieval:\n{question}")
            ]
        )
        try: 
            response = (prompt | self.llm_rewrite).invoke({"question": base})
            rephrased = getattr(response, 'content', '').strip()
            if not rephrased:
                logger.warning("⚠️ Rewrite returned empty response.")
                rephrased = base
            logger.info(f"📝 Rewritten question: {rephrased}")
            state['rephrased_question'] = rephrased
            state['rephrase_count'] = attempt
        except Exception as e:
            logger.exception(f"❌ Rewrite failed: {e}")
            state['rephrase_count'] = attempt
        return state

    def _node_classify_question(self, state: AgentState) -> AgentState:
        question = state['question'].content
        logger.debug(f"🔍 Classifying question: {question}")
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", CLASSIFICATION_SYSTEM_PROMPT),
                ("user", "Is this question on-topic for the gym knowledge base? Answer 'Yes' or 'No'.\n{question}")
            ]
        )
        try:
            response = (prompt | self.llm_chat).invoke({"question": question})
            classification = getattr(response, 'content', '').strip()
            state['on_topic'] = 'Yes' if parse_yes_no(classification) else 'No'
            state['rephrased_question'] = question
            logger.info(f"🧮 Classification result: {state['on_topic']}")
        except Exception as e:
            logger.exception(f"❌ Classification failed: {e}")
            state['on_topic'] = "No"
        return state

    def _node_retrieve_documents(self, state: AgentState) -> AgentState:
        if self.retriever is None:
            logger.error("❌ Retriever not initialized.")
            return state
        
        query = state.get('rephrased_question') or state['question'].content
        try:
            logger.debug(f"📚 Retrieving documents for query: {query}")
            docs = self.retriever.get_relevant_documents(query)
            logger.info(f"🔎 Retrieved {len(docs)} documents.")
            state['documents'] = docs
        except Exception as e:
            logger.exception(f"❌ Retrieval failed: {e}")
            state['documents'] = []
        return state

    def _node_grade_documents(self, state: AgentState) -> AgentState:
        docs: List[Document] = state.get('documents', [])
        query = state.get('rephrased_question') or state['question'].content
        if not docs:
            logger.warning("⚠️ No documents to grade.")
            state['proceed_to_generate'] = False
            return state
        
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", DOC_GRADE_SYSTEM_PROMPT),
                ("user", "Does this document help answer the question? Answer 'Yes' or 'No'.\nQuestion: {question}\nDocument: {document}")
            ]
        )
        kept: List[Document] = []
        for idx, d in enumerate(docs):
            snippet = d.page_content[:3000]
            try:
                res = (prompt | self.llm_chat).invoke({"question": query, "document": snippet})
                decision = getattr(res, 'content', '')
                keep = parse_yes_no(decision)
                logger.info(f"📄 Doc {idx} grading: {'Keep' if keep else 'Discard'}")
                if keep:
                    kept.append(d)
            except Exception as e:
                logger.exception(f"❌ Grading failed for doc {idx}: {e}")
        logger.info(f"✅ {len(kept)}/{len(docs)} documents kept after grading.")
        state['documents'] = kept
        state['proceed_to_generate'] = len(kept) > 0
        return state

    def _node_generate_answer(self, state: AgentState) -> AgentState:
        docs: List[Document] = state.get('documents', [])
        if not docs:
            logger.warning("⚠️ No documents to generate answer from.")
            state['messages'].append(AIMessage(content="I'm sorry, I don't have enough information to answer that question."))
            return state
        query = state.get('rephrased_question') or state['question'].content

        # build context with source tags
        context_parts = []
        for idx, d in enumerate(docs):
            source = d.metadata.get('source', f'Doc{idx+1}')
            part = f"[Doc {idx+1} | {source}]: \n{d.page_content[:1500]}"
            context_parts.append(part)
        context = "\n\n".join(context_parts)

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", ANSWER_SYSTEM_PROMPT),
                ("user", "Use the following documents to answer the question. Cite sources like [Doc 1], [Doc 2], etc. If insufficient info, say so briefly.\n\nDocuments:\n{context}\n\nQuestion: {question}")
            ]
        ) 
        try:
            res = (prompt | self.llm_chat).invoke({"context": context, "question": query})
            answer = getattr(res, 'content', '').strip()
            sources = {d.metadata.get('source', f'Doc{idx+1}') for idx, d in enumerate(docs)}
            if sources:
                answer += "\n\nSources: " + ", ".join(sorted(sources))
            state["messages"].append(AIMessage(content=answer))
            logger.debug(f"🗒️ Generated answer: {answer}")
        except Exception as e:
            logger.exception(f"❌ Answer generation failed: {e}")
            state['messages'].append(AIMessage(content="An error occurred while generating the answer."))
        return state

    def _node_generate_fallback(self, state: AgentState) -> AgentState:
        query = state['question'].content
        logger.debug(f"🔄 Generating fallback answer for query: {query}")
        msg = (
            "I could not find relevant information for your question: "
            f"'{query}'. Please rephrase or try another gym-related topic."
        )
        state["messages"].append(AIMessage(content=msg))
        return state

    def _node_off_topic(self, state: AgentState) -> AgentState:
        query = state['question'].content
        logger.debug(f"🚫 Off-topic response for query: {query}")
        msg = (
            "Your question appears outside the Jiten's Gym domain. "
            "Please ask about membership, facilities, trainers, classes, or related topics."
        )
        state["messages"].append(AIMessage(content=msg))
        return state

    # utility methods
    def _extract_last_ai_message(self, messages: List[BaseMessage]) -> str:
        for m in reversed(messages):
            if isinstance(m, AIMessage):
                return m.content
        return "No AI message found."

## main function

In [ ]:
CORPUS_DOCS: List[Document] = [
    Document(
        page_content="Jiten's Gym was founded in 2025 by former Olympic athlete Jiten Parmar. With over 15 years of experience in professional athletics, Jiten established the gym to provide personalized fitness solutions for people of all levels. The gym spans 10,000 square feet and features state-of-the-art equipment.",
        metadata={"source": "about.txt"}
    ),
    Document(
        page_content="Jiten's Gym is open Monday through Friday from 5:00 AM to 11:00 PM. On weekends, our hours are 7:00 AM to 9:00 PM. We remain closed on major national holidays. Members with Premium access can enter using their key cards 24/7, including holidays.",
        metadata={"source": "hours.txt"}
    ),
    Document(
        page_content="Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment; Standard (₹2,500/month) adds group classes and locker facilities; Premium (₹4,000/month) includes 24/7 access, personal training sessions, and spa facilities. We offer student and senior citizen discounts of 15% on all plans. Corporate partnerships are available for companies with 10+ employees joining.",
        metadata={"source": "membership.txt"}
    ),
    Document(
        page_content="Group fitness classes at Jiten's Gym include Yoga (beginner, intermediate, advanced), HIIT, Zumba, Spin Cycling, CrossFit, and Pilates. Beginner classes are held every Monday and Wednesday at 6:00 PM. Intermediate and advanced classes are scheduled throughout the week. The full schedule is available on our mobile app or at the reception desk.",
        metadata={"source": "classes.txt"}
    ),
    Document(
        page_content="Personal trainers at Jiten's Gym are all certified professionals with minimum 5 years of experience. Each new member receives a complimentary fitness assessment and one free session with a trainer. Our head trainer, Neha Kapoor, specializes in rehabilitation fitness and sports-specific training. Personal training sessions can be booked individually (₹800/session) or in packages of 10 (₹7,000) or 20 (₹13,000).",
        metadata={"source": "trainers.txt"}
    ),
    Document(
        page_content="Jiten's Gym's facilities include a cardio zone with 30+ machines, strength training area, functional fitness space, dedicated yoga studio, spin class room, swimming pool (25m), sauna and steam rooms, juice bar, and locker rooms with shower facilities. Our equipment is replaced or upgraded every 3 years to ensure members have access to the latest fitness technology.",
        metadata={"source": "facilities.txt"}
    ),
]

PERSIST_DIR = "db/rag"

In [ ]:
def main():
    config = RAGConfig()
    config.documents = CORPUS_DOCS
    pipeline = RAGPipeline(config=config, persist_dir=PERSIST_DIR, enable_vector_persist=True)
    if not pipeline.retriever:
        pipeline._build_vectorstore()
    
    pipeline.visualize()

    while True:
        user_input = input("\nAsk a question about Jiten's Gym (or type 'exit' to quit): ")
        logger.info(f"User input: {user_input}")
        if user_input.lower() in ['exit', 'quit']:
            logger.info("User exited the application.")
            print("Exiting. Goodbye!")
            break
        answer = pipeline.ask(user_input)
        print(f"\nAnswer: {answer}\n")
        logger.info(f"Answer: {answer}")

In [ ]:
main()